# モデル評価(myFM)

学習済みモデルの性能および特徴量影響度を確認するフェーズです。

## 実施内容
* 学習データの推論結果読込
* 各種メトリクス算出（AUC, Precision, Recall, F1, MCC, NDCG, Hit Rate）
* 評価結果のCSV出力

## 1. ライブラリインポート

In [ ]:
# 標準ライブラリ
import logging
import time
import warnings
from datetime import datetime
from functools import wraps
from pathlib import Path
from typing import Any, Callable, Dict


# サードパーティライブラリ
import numpy as np
import polars as pl
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
    ndcg_score,
)
from tqdm.auto import tqdm

# 自作モジュール
from utils_copy import(
    timing_decorator,
    get_latest_output_dir,
    load_data_pl,
    recall_at_k,
    precision_at_k,
    mcc_score,
    ndcg_at_k,
    hit_rate_at_k,
    compute_metrics,
    save_csvs
)


# ロガー設定
# python規約9.4のlogging_config.pyを参考に、loggerを設定
# INFO以上(INFO, WARNING, ERROR, CRITICAL)を出力
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

warnings.filterwarnings('default')

logger.info("ライブラリのインポート完了")


2026-06-10 02:35:41,871 - INFO - ライブラリのインポート完了


In [ ]:
# ==共通化== →utils.py
# 実行時間を記録

def timing_decorator(func: Callable) -> Callable:
    """関数の実行時間を計測してログ出力するデコレータ。"""

    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        start_time: float = time.time()
        try:
            result = func(*args, **kwargs)
        except Exception:
            logger.exception(f"{func.__name__} でエラーが発生しました")
            raise
        elapsed_time: float = time.time() - start_time
        logger.info(f"{func.__name__} 実行時間: {elapsed_time:.2f}秒")
        return result
    return wrapper

## 2. パス設定
* 入力・出力ファイルのパスを定義する
* 03で作成したタイムスタンプ付き出力ディレクトリを参照する
* モデル・パラメータを読み込み、評価結果の成果物を保存する

In [ ]:
# ==共通化== →utils.py
# 最新のoutputディレクトリを取得
@timing_decorator
def get_latest_output_dir(base_dir: str = "../output") -> Path:
    base = Path(base_dir)
    dirs = [d for d in base.iterdir() if d.is_dir()]
    if not dirs:
        raise FileNotFoundError("No output directories found.")
    latest = max(dirs, key=lambda d: d.name) 
    return latest

In [ ]:
# 入力パス
OUTPUT_DIR, OUTPUT_TIMESTAMP = get_latest_output_dir()
logger.info(f"使用する出力ディレクトリ: {OUTPUT_DIR}")
ARTIFACT_DIR: str = f"{OUTPUT_DIR}/artifacts"


# 入力
# 予測結果
PREDICTION_RESULT_FILE: str = (
    f"{OUTPUT_DIR}/prediction_result_{OUTPUT_TIMESTAMP}.csv"
)

# 出力
# 評価結果
EVALUATION_BASELINE_FILE: str = (
    f"{ARTIFACT_DIR}/evaluation_baseline_metrics_{OUTPUT_TIMESTAMP}.csv"
)
EVALUATION_TUNED_FILE: str = (
    f"{ARTIFACT_DIR}/evaluation_tuned_metrics_{OUTPUT_TIMESTAMP}.csv"
)
EVALUATION_COMPARISON_FILE: str = (
    f"{ARTIFACT_DIR}/evaluation_comparison_{OUTPUT_TIMESTAMP}.csv"
)

## 3. 変数定義
* 評価に使用する変数を定義

In [ ]:
# 評価指標(何番までで評価するか)
TOP_K = 10

# 閾値
threshold = 0.5

# データ読み込み時のチャンクサイズ
CHUNK_SIZE = 1000

## 4. データ読み込み

In [ ]:
# ==共通化== →utils.py
# データの読み込み（チャンク化して進捗バー表示）
@timing_decorator
def load_data_pl(path: str, chunk_size: int) -> pl.DataFrame:
    batches = pl.scan_csv(path).collect_batches(chunk_size=chunk_size)
    chunks = list(tqdm(batches))
    return pl.concat(chunks)

In [ ]:
# 予測結果の読み込み
df_predictions = load_data_pl(PREDICTION_RESULT_FILE, CHUNK_SIZE)

NameError: name 'OOF_FILE_BASELINE' is not defined

## 5. モデル評価
* 各評価指標の関数を定義
* 評価指標の計算
* 評価指標の保存


### 5.1 各評価指標の関数

In [ ]:
# 共通化
# モジュール化候補
@timing_decorator
def recall_at_k(y_true: np.ndarray, y_pred_proba: np.ndarray, k: int) -> float:
    """Recall@Kを計算

    Args:
        y_true: 真のラベル
        y_score: 予測スコア
        k: 上位K件

    Returns:
        Recall@K
    """
    idx: np.ndarray = np.argsort(-y_pred_proba)[:k]
    return float(y_true[idx].sum() / y_true.sum())


@timing_decorator
def precision_at_k(y_true: np.ndarray, y_pred_proba: np.ndarray, k: int) -> float:
    """Precision@Kを計算

    Args:
        y_true: 真のラベル
        y_score: 予測スコア
        k: 上位K件

    Returns:
        Precision@K
    """
    idx: np.ndarray = np.argsort(-y_pred_proba)[:k]
    return float(y_true[idx].sum() / k)


@timing_decorator
def mcc_score(y_true: np.ndarray, y_pred_label: np.ndarray) -> float:
    """Matthews Correlation Coefficientを計算

    Args:
        y_true: 真のラベル
        y_pred: 予測ラベル

    Returns:
        MCC
    """
    return float(matthews_corrcoef(y_true, y_pred_label))


@timing_decorator
def ndcg_at_k(y_true: np.ndarray, y_pred_proba: np.ndarray, k: int) -> float:
    """NDCG@Kを計算

    Args:
        y_true: 真のラベル
        y_score: 予測スコア
        k: 上位K件

    Returns:
        NDCG@K
    """
    y_true_2d: np.ndarray = y_true.reshape(1, -1)
    y_score_2d: np.ndarray = y_pred_proba.reshape(1, -1)
    return float(ndcg_score(y_true_2d, y_score_2d, k=k))


@timing_decorator
def hit_rate_at_k(y_true: np.ndarray, y_pred_proba: np.ndarray, k: int) -> float:
    """Hit Rate@Kを計算

    Args:
        y_true: 真のラベル
        y_score: 予測スコア
        k: 上位K件

    Returns:
        Hit Rate@K
    """
    idx: np.ndarray = np.argsort(-y_pred_proba)[:k]
    return 1.0 if y_true[idx].sum() > 0 else 0.0

### 5.2 評価関数の計算
* y, y_pred_label, y_pred_probaを入力とし、metricsの辞書を返す

In [ ]:
# 共通化→utils.py

@timing_decorator
def compute_metrics(
    y: np.ndarray,
    y_pred_proba: np.ndarray,
    y_pred_label: np.ndarray,
    top_k: int
) -> Dict[str, Any]:

    metrics: Dict[str, Any] = {
        "auc": roc_auc_score(y, y_pred_proba),
        f"recall_at_{top_k}": recall_at_k(y, y_pred_proba, top_k),
        f"precision_at_{top_k}": precision_at_k(y, y_pred_proba, top_k),
        "mcc": mcc_score(y, y_pred_label),
        f"ndcg_at_{top_k}": ndcg_at_k(y, y_pred_proba, top_k),
        f"hit_rate_at_{top_k}": hit_rate_at_k(y, y_pred_proba, top_k),
    }

    return metrics

In [ ]:
# baselineモデル評価
y: np.ndarray = df_predictions["y"].to_numpy()

y_pred_proba_baseline: np.ndarray = df_predictions["y_baseline_proba"].to_numpy()
y_pred_label_baseline: np.ndarray = df_predictions["y_baseline_pred"].to_numpy()

# threshold（必要なら）
y_pred_binary_baseline: np.ndarray = (y_pred_proba_baseline > threshold).astype(int)

In [ ]:
# baselineモデル評価
metrics_baseline = compute_metrics(
    y=y,
    y_pred_proba=y_pred_proba_baseline,
    y_pred_label=y_pred_label_baseline,
    top_k=TOP_K
)

2026-06-10 23:35:08,357 - INFO - recall_at_k 実行時間: 0.00秒
2026-06-10 23:35:08,357 - INFO - precision_at_k 実行時間: 0.00秒
2026-06-10 23:35:08,365 - INFO - mcc_score 実行時間: 0.01秒
2026-06-10 23:35:08,368 - INFO - ndcg_at_k 実行時間: 0.00秒
2026-06-10 23:35:08,368 - INFO - hit_rate_at_k 実行時間: 0.00秒
2026-06-10 23:35:08,369 - INFO - compute_metrics 実行時間: 0.06秒


In [ ]:
# tunedモデル評価
y: np.ndarray = df_predictions["y"].to_numpy()

y_pred_proba_tuned: np.ndarray = df_predictions["y_tuned_proba"].to_numpy()
y_pred_label_tuned: np.ndarray = df_predictions["y_tuned_pred"].to_numpy()

# threshold（必要なら）
y_pred_binary_tuned: np.ndarray = (y_pred_proba_tuned > threshold).astype(int)

In [ ]:
metrics_tuned = compute_metrics(
    y=y,
    y_pred_proba=y_pred_proba_tuned,
    y_pred_label=y_pred_label_tuned,
    top_k=TOP_K
)

2026-06-10 23:35:13,876 - INFO - recall_at_k 実行時間: 0.00秒
2026-06-10 23:35:13,910 - INFO - precision_at_k 実行時間: 0.01秒
2026-06-10 23:35:13,963 - INFO - mcc_score 実行時間: 0.05秒
2026-06-10 23:35:13,972 - INFO - ndcg_at_k 実行時間: 0.00秒
2026-06-10 23:35:13,973 - INFO - hit_rate_at_k 実行時間: 0.00秒
2026-06-10 23:35:13,973 - INFO - compute_metrics 実行時間: 0.11秒


### 5.3 評価結果の保存

In [ ]:
@timing_decorator

# pd・pl関係なくdfをCSVファイルに保存する
def save_csvs(files: Dict[str, pd.DataFrame | pl.DataFrame]) -> None:
    for file_path, df in files.items():

        # Polars DataFrame
        if isinstance(df, pl.DataFrame):
            df.write_csv(file_path)

        # pandas DataFrame
        elif isinstance(df, pd.DataFrame):
            df.to_csv(file_path, index=False)

        else:
            raise TypeError(f"Unsupported type: {type(df)}")

        logger.info("%s 保存完了", file_path)

In [ ]:
df_metrics_baseline = pl.DataFrame([metrics_baseline])
df_metrics_tuned = pl.DataFrame([metrics_tuned])
df_metrics_comparison = pl.concat([df_metrics_baseline, df_metrics_tuned])
save_csvs({
    EVALUATION_BASELINE_FILE: df_metrics_baseline,
    EVALUATION_TUNED_FILE: df_metrics_tuned,
    EVALUATION_COMPARISON_FILE: df_metrics_comparison,
})

2026-06-10 23:35:18,532 - INFO - ../output/2606101938/artifacts/evaluation_baseline_metrics_2606101938.csv 保存完了
2026-06-10 23:35:18,534 - INFO - ../output/2606101938/artifacts/evaluation_tuned_metrics_2606101938.csv 保存完了
2026-06-10 23:35:18,538 - INFO - ../output/2606101938/artifacts/evaluation_comparison_2606101938.csv 保存完了
2026-06-10 23:35:18,538 - INFO - save_csvs 実行時間: 0.02秒
